# WISER Module 4 Colab Template: Making Meaning with NLP Semantics

        **Learning objective:** Extract, structure, compare, and summarize addiction-related text while evaluating output quality, bias, terminology drift, and limitations.

> **Before you begin:** In Google Colab, choose **File → Save a copy in Drive**. Work only in your copy.
>
> This notebook connects to your prior WISER work. If you completed earlier modules, you may import outputs from those notebooks where prompted.
>
> **Privacy reminder:** Do not upload protected health information, identifiable clinical notes, or non-approved institutional data. Use the WISER synthetic datasets unless your instructor has explicitly approved another dataset.

In [ ]:
MODULE_ID = "Module 4"
NOTEBOOK_TITLE = "WISER Module 4: NLP Semantics"

## Setup
Run this cell first. It loads shared helper functions used throughout the notebook.

In [ ]:
from pathlib import Path
import json
import textwrap
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PALETTE = {
    "teal": "#20808D",
    "rust": "#A84B2F",
    "dark_teal": "#1B474D",
    "light_cyan": "#BCE2E7",
    "mauve": "#944454",
    "gold": "#FFC553",
}

STIGMA_TERMS = {
    "addict": "person with a substance use disorder",
    "addicts": "people with substance use disorders",
    "abuser": "person with a substance use disorder",
    "drug abuse": "substance use or substance use disorder",
    "clean": "negative toxicology / not currently using",
    "dirty": "positive toxicology",
    "junkie": "person with a substance use disorder",
}

def stigma_audit(text):
    '''Return potentially stigmatizing terms found in a string.'''
    text_lower = str(text).lower()
    return {term: suggestion for term, suggestion in STIGMA_TERMS.items() if term in text_lower}

def safe_label(text):
    '''Lightweight learner-facing label checker. Human review is still required.'''
    hits = stigma_audit(text)
    if hits:
        print("Review this wording:")
        for term, suggestion in hits.items():
            print(f" - Replace '{term}' with '{suggestion}' when clinically appropriate.")
    return text

def quick_profile(df, name="dataset"):
    '''Basic data profile for WISER learner notebooks.'''
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(1),
        "unique_n": df.nunique(dropna=True),
    })
    display(profile)
    display(df.head())

## Section 1: Data Loading and Exploration

This template uses a synthetic clinical-note corpus if no course CSV is available. Replace `DATA_URL` with the final WISER file URL when publishing.

In [ ]:
DATA_URL = ""  # TODO: paste final WISER synthetic_notes.csv URL, if available

if DATA_URL:
    notes = pd.read_csv(DATA_URL)
else:
    notes = pd.DataFrame({
        "note_id": range(1, 9),
        "year": [2014, 2015, 2017, 2018, 2020, 2021, 2022, 2024],
        "setting": ["clinic", "survey", "clinic", "public_health", "clinic", "survey", "clinic", "public_health"],
        "text": [
            "Patient reports nonmedical opioid use and limited access to MOUD.",
            "Respondent endorses pain reliever misuse in prior year.",
            "Person with OUD reports transportation barriers and unstable housing.",
            "Heroin use variable renamed in surveillance extract; definition changed from prior wave.",
            "Patient denies opioid use but reports cravings and missed appointments.",
            "Participant reports fentanyl exposure and interest in treatment.",
            "Person receiving buprenorphine reports improved functioning.",
            "Pain reliever dependence field crosswalked to DSM-5 OUD severity proxy."
        ],
        "substance_class": ["opioid", "opioid", "opioid", "opioid", "opioid", "synthetic_opioid", "opioid", "opioid"]
    })

quick_profile(notes, "synthetic_notes")
notes["text"].apply(stigma_audit).tolist()

### Stop and Reflect

Which text fields are likely to reflect patient language, clinician language, or survey-instrument language? Why does that distinction matter?

## Section 2: Core Method Application — Text Preprocessing

In [ ]:
import re

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\\s\\-]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

notes["text_clean"] = notes["text"].map(normalize_text)
notes[["note_id", "text", "text_clean"]]

## Section 3: Semantic Similarity

The preferred version uses sentence embeddings. If `sentence_transformers` is unavailable, the fallback uses TF-IDF so learners can still complete the exercise.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    X = model.encode(notes["text_clean"].tolist())
    method = "sentence-transformers/all-MiniLM-L6-v2"
except Exception as e:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    X = vectorizer.fit_transform(notes["text_clean"]).toarray()
    method = "TF-IDF fallback"

sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim, index=notes["note_id"], columns=notes["note_id"])
print("Embedding method:", method)
display(sim_df.round(2))

## Section 4: Terminology Crosswalk Exercise

In [ ]:
crosswalk = pd.DataFrame({
    "legacy_term": ["pain reliever dependence", "opioid misuse", "heroin use", "substance abuse"],
    "preferred_or_current_term": ["opioid use disorder severity proxy", "nonmedical use of prescription opioids", "heroin use", "substance use disorder"],
    "risk_note": [
        "May not map cleanly to DSM-5 severity.",
        "Survey wording changes can affect trend interpretation.",
        "Substance class may shift with fentanyl co-exposure.",
        "Stigmatizing and outdated in many contexts."
    ]
})
display(crosswalk)

for label in crosswalk["legacy_term"]:
    safe_label(label)

## Section 5: Foundational Pathway Task

Write a 150–250 word memo explaining one terminology mismatch you found and how it could affect interpretation of addiction surveillance data.

In [ ]:
memo_foundational = """TODO: Write your memo here."""
print(textwrap.fill(memo_foundational, 90))

## Section 6: Applied Pathway Extension

Modify the reference terms below, compute their nearest notes, and decide whether the matches are clinically sensible.

In [ ]:
reference_terms = [
    "person with opioid use disorder seeking treatment",
    "transportation barrier to medication appointment",
    "fentanyl exposure and overdose risk",
]

# Applied task: encode reference_terms with the same method and compute nearest notes.
# TODO: complete this cell if you are on the Applied Pathway.
print("Applied extension placeholder:", reference_terms)

## Section 7: Ethical Reflection

1. Could this workflow expose or reconstruct sensitive information?
2. Which terminology choices might encode stigma?
3. What documentation is needed so another researcher can reuse your crosswalk?

## Section 8: FAIR Export and Submission

In [ ]:
metadata = {
    "module": MODULE_ID,
    "notebook": NOTEBOOK_TITLE,
    "created_by": "WISER learner",
    "uses_synthetic_or_public_data": True,
    "contains_phi": False,
    "fair_notes": {
        "findable": "Outputs saved with clear names in outputs/.",
        "accessible": "Notebook and exported files can be shared through the course portal.",
        "interoperable": "CSV/JSON/Markdown formats used where possible.",
        "reusable": "README and metadata describe inputs, assumptions, and outputs.",
    },
}

(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
(OUTPUT_DIR / "README.md").write_text(f'''# {NOTEBOOK_TITLE} Outputs

This folder contains learner-generated outputs for {MODULE_ID}.

## Files

- `metadata.json`: FAIR-oriented metadata
- Additional module-specific artifacts produced by notebook cells

## Privacy

This template is designed for synthetic or public data. Do not include PHI or identifiable institutional data.
''')

print("FAIR export complete.")
print("Generated:", sorted(str(p) for p in OUTPUT_DIR.iterdir()))